# Lecture 36 Worksheet (v2): CI/CD Basics
**NAVTTC / NIAI AI/ML Trainer Cohort**. Runs in Colab or Jupyter, plain Python only. Fill every `None` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or `FIX`.

Running example: a **fraud detection** system. Part A is in this notebook. Part B is on GitHub (your first real pipeline).

In [ ]:
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("ready")

## Task 1: Put the stages in order

Fill the list with the SDLC stages in the right order: develop, build, test, release. Then say which stages are CI and which are CD.

In [ ]:
stages = []              # YOUR CODE: four words, in order
ci = []                  # YOUR CODE: the two CI stages
cd = []                  # YOUR CODE: the CD stage

In [ ]:
check("order", stages == ["develop", "build", "test", "release"], "develop, build, test, release")
check("CI", sorted(ci) == ["build", "test"], "CI = build + test")
check("CD", cd == ["release"], "CD = release")

## Task 2: Write a function and its tests

Our fraud function gives a risk score from 0 to 100.
- Start with `20 * (amount / usual_amount)`.
- Add 30 if `new_country` is True.
- Never go above 100 (use `min`).

Then write three **unit tests** as functions starting with `test_` that use `assert`: a normal payment (amount equal to usual, same country) scores below 30; a payment 10 times usual scores 100; the same normal payment from a new country scores more than the same payment at home.

In [ ]:
def risk_score(amount, usual_amount, new_country=False):
    return None            # YOUR CODE

def test_normal_is_low():
    pass                   # YOUR CODE: assert ...

def test_huge_is_capped_at_100():
    pass                   # YOUR CODE

def test_new_country_adds_risk():
    pass                   # YOUR CODE

In [ ]:
tests = [test_normal_is_low, test_huge_is_capped_at_100, test_new_country_adds_risk]
passed = 0
for t in tests:
    try:
        t(); passed += 1
    except AssertionError:
        print("FAILED:", t.__name__)
check(f"{passed}/3 tests pass", passed == 3 and risk_score(100, 100) == 20, "check the formula and the asserts")
print("These same lines, saved as risk.py and test_risk.py, are what the GitHub workflow will run.")

## Task 3: A mini pipeline that stops on failure

A pipeline runs stages in order and **stops at the first failure**. That is the guard rail.

Write `run_pipeline(stages)`. `stages` is a list of `(name, function)` pairs; each function returns True (pass) or False (fail). For each stage print `name: passed` or `name: FAILED`. If a stage fails, stop and return the name of the failed stage. If all pass, return `None`.

In [ ]:
def run_pipeline(stages):
    pass                   # YOUR CODE

In [ ]:
good = [("build", lambda: True), ("test", lambda: True), ("deploy", lambda: True)]
bad = [("build", lambda: True), ("test", lambda: False), ("deploy", lambda: True)]
print("-- good pipeline"); r1 = run_pipeline(good)
print("-- bad pipeline");  r2 = run_pipeline(bad)
check("good pipeline passes", r1 is None, "return None when all pass")
check("bad pipeline stops at test", r2 == "test", "deploy must NOT run after test fails")

## Task 4: Delivery or deployment?

Change `run_pipeline` into `run_release(stages, approve, auto_deploy)` for the release step only. After all stages pass:
- if `auto_deploy` is True: return `"deployed to prod"` (continuous deployment).
- else if `approve` is True: return `"deployed to prod"` (continuous delivery, a person approved).
- else return `"waiting for approval"`.
If a stage fails, return `"stopped at <name>"`.

In [ ]:
def run_release(stages, approve=False, auto_deploy=False):
    return None            # YOUR CODE

In [ ]:
check("delivery waits", run_release(good) == "waiting for approval", "no approval and no auto_deploy")
check("delivery after approval", run_release(good, approve=True) == "deployed to prod", "")
check("deployment is automatic", run_release(good, auto_deploy=True) == "deployed to prod", "")
check("failure blocks both", run_release(bad, auto_deploy=True) == "stopped at test", "")

## Task 5: Canary release, a small simulation

A canary sends a small share of requests to the new version. Suppose the **new** version has a bug that makes 20% of its requests fail, and the **old** one never fails. We send 10% of 1000 requests to the new version.

Complete `canary(n, share, bug_rate, seed)`: for each request, choose new with probability `share`; if new, it fails with probability `bug_rate`. Return `(failed_requests, requests_sent_to_new)`. Compare with sending everyone to the new version (`share=1.0`).

In [ ]:
import random
def canary(n=1000, share=0.1, bug_rate=0.2, seed=1):
    rng = random.Random(seed)
    failed = to_new = 0
    for _ in range(n):
        pass               # YOUR CODE
    return failed, to_new

In [ ]:
f10, n10 = canary(share=0.1)
f100, n100 = canary(share=1.0)
print(f"canary 10%:  {f10} failed requests out of 1000 ({n10} went to the new version)")
print(f"everyone:    {f100} failed requests out of 1000")
check("canary hurts fewer users", f10 < f100 / 3, "compare the two numbers")
R10, R100 = f10, f100

## Task 6: Read a workflow (no code)
Answer in the cells below as plain strings.

```yaml
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install pytest
      - run: pytest
```

In [ ]:
q1 = ""   # When does this workflow run?
q2 = ""   # Which step runs the tests?
q3 = ""   # If pytest finds a failing test, what does GitHub show, and do later jobs run?

## Part B: your first pipeline on GitHub (do this in Hour 2)
1. Create a new public repository with a README.
2. Add a file `risk.py` containing your `risk_score` function.
3. Add `test_risk.py`:
```python
from risk import risk_score
def test_normal_is_low(): assert risk_score(100, 100) < 30
def test_huge_is_capped_at_100(): assert risk_score(1000, 100) == 100
def test_new_country_adds_risk(): assert risk_score(100, 100, True) > risk_score(100, 100, False)
```
4. Add `.github/workflows/ci.yml` (the file from slide 14). Commit. Open the **Actions** tab: you should see a green tick.
5. Change `20 *` to `2 *` in `risk.py`, commit, and watch it turn red. Open the log and find the failed test. Fix it and watch it turn green.
6. Screenshot your red and your green run for Hour 3.

**Reflection (2 sentences each):** What did the red cross save you from? Would you choose delivery or deployment for this fraud system, and why?